### SoRL v3 Kernel — Minimal Training Notebook

**Loss**: `loss = alpha_traj * traj_loss + alpha_contrastive * hinge_loss + alpha_abs * abs_loss`
- `traj_loss`: CE on NL positions p(s|a)
- `hinge_loss`: margin between clean and corrupted abstract sequences
- `abs_loss`: CE on abstract positions p(a|s)

In [3]:
# Test with NL token replacement, instead of just NL token masking
# Test with random K training, just to see if randomizing K beats static K, even if we evaluate static K conditioning accuracy

In [4]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper

from sorl.trainer_ablate import SoRLTrainerv2, SoRLTrainerv3
from sorl.trainer_ablate import SoRLConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [5]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
# ============================================================
# SoRL v3 Training Kernel — all logic inline, easy to hack
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import time, os, json
from sorl.sorl_trainer import sorl_search, corrupt_abstract_tokens, ortho_loss
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

# ---- Config (edit freely) ----
cfg = dict(
    # Model
    model_name   = "Qwen/Qwen3-0.6B",
    abs_vocab    = 128,

    # Data
    dataset      = "gsm8k",
    max_length   = 256,
    batch_size   = 2,

    # SoRL search
    K            = 4,
    num_rollouts = 4,
    max_iters    = 2,
    temperature  = 1.0,
    mem_span_abs = 1792,
    mem_span_traj= 1792,

    # Corruption
    corrupt_method = "shuffle",
    corrupt_ratio  = 0.3,

    # Loss weights
    alpha_traj       = 1.0,     # p(s|a)
    alpha_contrastive= 1.0,     # hinge loss
    alpha_abs        = 0.5,     # p(a|s)
    gamma            = 0.5,     # hinge margin

    # Randomization (set to None to disable, or give a value/range)
    random_K         = None,    # e.g. (2, 4, 6, 8) — choices for K per batch
    strip_suffix     = None,    # e.g. (0.1, 1.0) — keep_frac range
    compress_prefix  = None,    # e.g. (0.0, 0.8) — compress_frac range
    random_mem_span  = None,    # e.g. (64, 1792) — memory_span_abs range

    # Optimizer
    lr           = 1e-5,
    emb_lr_mult  = 1.0,
    weight_decay = 0.01,
    max_grad_norm= 1.0,
    warmup_steps = 50,
    cooldown_frac= 0.4,

    # Training
    num_epochs   = 3,
    grad_accum   = 4,
    log_every    = 10,
    eval_every   = 99999,
    eval_samples = 100,
    save_every   = 99999,
    output_dir   = "./ckpt/v3_test",
)

print("Config:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

Config:
  model_name: Qwen/Qwen3-0.6B
  abs_vocab: 128
  dataset: gsm8k
  max_length: 256
  batch_size: 2
  K: 4
  num_rollouts: 4
  max_iters: 2
  temperature: 1.0
  mem_span_abs: 1792
  mem_span_traj: 1792
  corrupt_method: shuffle
  corrupt_ratio: 0.3
  alpha_traj: 1.0
  alpha_contrastive: 1.0
  alpha_abs: 0.5
  alpha_soft_zipf: 2.0
  alpha_ortho: 0.0
  alpha_anchor: 1.0
  gamma: 0.5
  warmup_anchor_steps: 200
  lr: 1e-05
  emb_lr_mult: 1.0
  weight_decay: 0.01
  max_grad_norm: 1.0
  warmup_steps: 50
  cooldown_frac: 0.4
  num_epochs: 3
  grad_accum: 4
  log_every: 10
  eval_every: 99999
  eval_samples: 100
  save_every: 99999
  output_dir: ./ckpt/v3_anchor_warmup


In [7]:
# ============================================================
# Model + Data + Optimizer setup
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SorlModelWrapper.from_pretrained(cfg["model_name"], abstract_vocab_size_list=[cfg["abs_vocab"]])
model = model.to(device).train()
tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
pad_token_id = tokenizer.pad_token_id
base_vocab = int(model.vocab_sizes[0].item())
total_vocab = int(model.vocab_sizes.sum().item())

train_ds = get_dataset(cfg["dataset"], split="train", tokenizer=tokenizer, max_length=cfg["max_length"])
val_ds   = get_dataset(cfg["dataset"], split="test",  tokenizer=tokenizer, max_length=cfg["max_length"])
dl = torch.utils.data.DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate_fn, num_workers=0)

# Separate param groups for embedding LR multiplier
emb_params, other_params = [], []
for name, p in model.named_parameters():
    if "embed_tokens" in name or "lm_head" in name:
        emb_params.append(p)
    else:
        other_params.append(p)
optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": cfg["lr"]},
    {"params": emb_params,   "lr": cfg["lr"] * cfg["emb_lr_mult"]},
], weight_decay=cfg["weight_decay"])

total_steps = len(dl) * cfg["num_epochs"] // cfg["grad_accum"]
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Steps/epoch: {len(dl)} | Total steps: {total_steps}")
print(f"Effective batch: {cfg['batch_size'] * cfg['grad_accum']}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Train: 7473 | Val: 1319 | Steps/epoch: 3737 | Total steps: 2802
Effective batch: 8


In [ ]:
# ---- Experimental Kernel, don't delete ----
for batch in dl: 
    break

input_ids     = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
prompt_len    = batch["prompt_len"].to(device)

# Idea 1. Randomize abstraction ratio 'K' (keep the same within each batch)
# Idea 2. Randomize prefix length within which we insert abstract tokens (keep the same within each batch)
# Idea 3. Randomly replace NL token with abstract tokens

with torch.no_grad():
    best_data, _, _, exp_attn, exp_pl = sorl_search(
        model, input_ids, attention_mask, prompt_len, pad_token_id,
        n=cfg["num_rollouts"], K=cfg["K"], max_iterations=cfg["max_iters"],
        memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"],
        temperature=cfg["temperature"],
    )

In [62]:
# ============================================================
# 5 Randomization Ideas for SoRL v3 Training
# ============================================================
import random

# ==== 1. Random K — vary abstraction granularity per batch ====
def sample_random_K(K_choices=(2, 4, 6, 8)):
    """Sample a random abstraction ratio K for this batch.
    Smaller K = finer abstractions, larger K = coarser.
    Use BEFORE sorl_search: sorl_search(..., K=sample_random_K())
    """
    return random.choice(K_choices)


# ==== 2. Strip suffix abstractions — keep abs only in a random prefix ====
def drop_abs_suffix(best_data, exp_attn, exp_pl, base_vocab, pad_token_id, keep_frac=None):
    """Keep abstract tokens only in a random prefix of the response.
    Beyond the cutoff, remove abstract tokens (NL-only suffix).

    [p p | a t t t a t t t a t t t] --keep_frac=0.5--> [p p | a t t t t t t]
                  ^cutoff

    Args:
        keep_frac: float in (0,1], fraction of response to keep abstractions.
                   If None, sampled uniformly from (0.1, 1.0].
    Returns:
        new_data, new_attn, new_pl: re-padded sequences
    """
    if keep_frac is None:
        keep_frac = random.uniform(0.1, 1.0)

    B, L = best_data.shape
    new_seqs, new_pls = [], []
    for b in range(B):
        pl = exp_pl[b].item() if isinstance(exp_pl, torch.Tensor) else exp_pl[b]
        valid_len = int(exp_attn[b].sum().item())
        seq = best_data[b, :valid_len]
        prompt = seq[:pl]
        response = seq[pl:]

        cutoff = max(1, int(len(response) * keep_frac))
        prefix = response[:cutoff]                          # keep as-is (NL + abs)
        suffix_nl = response[cutoff:][response[cutoff:] < base_vocab]  # strip abs

        new_seq = torch.cat([prompt, prefix, suffix_nl])
        new_seqs.append(new_seq)
        new_pls.append(pl)

    max_len = max(s.shape[0] for s in new_seqs)
    new_data = torch.full((B, max_len), pad_token_id, device=best_data.device, dtype=best_data.dtype)
    new_attn = torch.zeros((B, max_len), device=best_data.device, dtype=exp_attn.dtype)
    for b, s in enumerate(new_seqs):
        new_data[b, :s.shape[0]] = s
        new_attn[b, :s.shape[0]] = 1
    new_pl = torch.tensor(new_pls, device=best_data.device)
    return new_data, new_attn, new_pl


# ==== 3. Compress prefix chunks — drop NL, keep only abstract in prefix ====
def drop_nl_prefix(best_data, exp_attn, exp_pl, base_vocab, pad_token_id, compress_frac=None):
    """In a random prefix of the response, drop NL tokens and keep only abstract tokens.

    [p p | a t t t a t t t a t t t a t t t]
    compress_frac=0.75 --> [p p | a a a  a t t t]   (3 chunks → abs only, last chunk intact)
    compress_frac=0.50 --> [p p | a a  a t t t a t t t]
    compress_frac=0.25 --> [p p | a  a t t t a t t t a t t t]

    Args:
        compress_frac: float in [0,1), fraction of response to compress.
                       If None, sampled uniformly from [0.0, 0.8).
    Returns:
        new_data, new_attn, new_pl: re-padded compressed sequences
    """
    if compress_frac is None:
        compress_frac = random.uniform(0.0, 0.8)

    B, L = best_data.shape
    new_seqs, new_pls = [], []
    for b in range(B):
        pl = exp_pl[b].item() if isinstance(exp_pl, torch.Tensor) else exp_pl[b]
        valid_len = int(exp_attn[b].sum().item())
        seq = best_data[b, :valid_len]
        prompt = seq[:pl]
        response = seq[pl:]

        cutoff = int(len(response) * compress_frac)
        prefix_abs = response[:cutoff][response[:cutoff] >= base_vocab]  # abs only
        suffix = response[cutoff:]                                        # keep all

        new_seq = torch.cat([prompt, prefix_abs, suffix])
        new_seqs.append(new_seq)
        new_pls.append(pl)

    max_len = max(s.shape[0] for s in new_seqs)
    new_data = torch.full((B, max_len), pad_token_id, device=best_data.device, dtype=best_data.dtype)
    new_attn = torch.zeros((B, max_len), device=best_data.device, dtype=exp_attn.dtype)
    for b, s in enumerate(new_seqs):
        new_data[b, :s.shape[0]] = s
        new_attn[b, :s.shape[0]] = 1
    new_pl = torch.tensor(new_pls, device=best_data.device)
    return new_data, new_attn, new_pl


# ==== 4. Random memory_span_abs — force abstract dependency ====
def sample_random_memory_span(lo=64, hi=1792):
    """Sample a random memory_span_abs for this batch.
    Smaller spans limit how far back the model attends to NL tokens,
    forcing it to actually use abstract tokens as information carriers
    rather than just as attention routers to nearby NL tokens.
    """
    return random.randint(lo, hi)


# ---- Quick demo on the batch from cell 6 ----
print("=== Demo: 5 randomization functions ===\n")

# 1. Random K
K_sampled = sample_random_K()
print(f"1. Random K: {K_sampled}")

# 2. Strip suffix abstractions
d2, a2, p2 = drop_abs_suffix(best_data, exp_attn, exp_pl, base_vocab, pad_token_id, keep_frac=0.5)
n_abs_before = (best_data >= base_vocab).sum().item()
n_abs_after  = (d2 >= base_vocab).sum().item()
print(f"2. Drop abstraction suffix (keep_frac=0.5): shape {best_data.shape} → {d2.shape}, abs tokens {n_abs_before} → {n_abs_after}")

# 3. Compress prefix chunks
d3, a3, p3 = drop_nl_prefix(best_data, exp_attn, exp_pl, base_vocab, pad_token_id, compress_frac=0.5)
n_nl_before = ((best_data < base_vocab) & (best_data != pad_token_id)).sum().item()
n_nl_after  = ((d3 < base_vocab) & (d3 != pad_token_id)).sum().item()
print(f"3. Drop NL prefix (frac=0.5): shape {best_data.shape} → {d3.shape}, NL tokens {n_nl_before} → {n_nl_after}")

# 4. Random memory_span_abs
span = sample_random_memory_span(lo=64, hi=512)
print(f"4. Random memory_span_abs: {span}")

print("\nAll functions ready for integration into training loop.")

=== Demo: 5 randomization functions ===

1. Random K: 4
2. Drop abstraction suffix (keep_frac=0.5): shape torch.Size([2, 307]) → torch.Size([2, 241]), abs tokens 95 → 62
3. Drop NL prefix (frac=0.5): shape torch.Size([2, 307]) → torch.Size([2, 185]), NL tokens 385 → 251
4. Random memory_span_abs: 359

All functions ready for integration into training loop.


In [ ]:
# ============================================================
# Plot Loss Curves
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("SoRL v3 — Training Curves", fontsize=14)

plots = [
    ("loss", "Total Loss"),
    ("base_loss", "Base Traj Loss (no grad)"),
    ("traj_loss", "Traj Loss p(s|a)"),
    ("hinge_loss", "Hinge Contrastive Loss"),
    ("abs_loss", "Abstract Loss p(a|s)"),
    ("lr", "Learning Rate"),
]

for ax, (key, title) in zip(axes.flat, plots):
    if key in history and len(history[key]) > 0:
        ax.plot(history["step"], history[key], linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("step")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{title} (no data)")

plt.tight_layout()
plt.show()

In [ ]:
# Randomization config guide:
# - random_K = (2, 4, 6, 8)       → vary chunk granularity per batch
# - strip_suffix = (0.1, 1.0)     → keep abs in random prefix only
# - compress_prefix = (0.0, 0.8)  → compress prefix chunks to abs-only
# - random_mem_span = (64, 1792)  → vary memory span to force abs dependency
#
# Set any to None in cfg to disable. Ranges are (lo, hi) for uniform sampling.

In [ ]:
# ============================================================
# Training loop — Full v3 with 4 randomizations
# ============================================================
os.makedirs(cfg["output_dir"], exist_ok=True)
history = {"step": [], "loss": [], "base_loss": [], "traj_loss": [],
           "hinge_loss": [], "abs_loss": [], "lr": []}

model.train()
global_step = 0
accum_count = 0
t_start = time.time()

for epoch in range(cfg["num_epochs"]):
    for batch_idx, batch in enumerate(dl):
        input_ids     = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prompt_len    = batch["prompt_len"].to(device)

        # ---- LR schedule ----
        lr = get_lr(global_step)
        optimizer.param_groups[0]["lr"] = lr
        optimizer.param_groups[1]["lr"] = lr * cfg["emb_lr_mult"]

        # ---- Randomization 1: Random K ----
        K_this = sample_random_K(cfg["random_K"]) if cfg["random_K"] else cfg["K"]

        # ---- Randomization 4: Random memory_span_abs ----
        if cfg["random_mem_span"]:
            mem_abs = sample_random_memory_span(*cfg["random_mem_span"])
        else:
            mem_abs = cfg["mem_span_abs"]

        # ---- Base traj loss (logging only, no grad) ----
        with torch.no_grad():
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            si = torch.arange(labels.size(1), device=device).unsqueeze(0)
            labels[si < prompt_len.unsqueeze(1)] = -100
            out = model(input_ids=input_ids, attention_mask=attention_mask,
                        memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"])
            lg = out.logits.clone()
            lg[:, :, base_vocab:] = -float("inf")
            base_loss = nn.CrossEntropyLoss(ignore_index=-100)(
                lg[:, :-1].contiguous().view(-1, lg.size(-1)),
                labels[:, 1:].contiguous().view(-1)
            )
            del out, lg

        # ---- SoRL search (no grad) ----
        with torch.no_grad():
            best_data, _, _, exp_attn, exp_pl = sorl_search(
                model, input_ids, attention_mask, prompt_len, pad_token_id,
                n=cfg["num_rollouts"], K=K_this, max_iterations=cfg["max_iters"],
                memory_span_abs=mem_abs, memory_span_traj=cfg["mem_span_traj"],
                temperature=cfg["temperature"],
            )

        # ---- Randomization 2: Strip suffix abstractions ----
        if cfg["strip_suffix"]:
            frac = random.uniform(*cfg["strip_suffix"])
            best_data, exp_attn, exp_pl = drop_abs_suffix(
                best_data, exp_attn, exp_pl, base_vocab, pad_token_id, keep_frac=frac)

        # ---- Randomization 3: Compress prefix chunks ----
        if cfg["compress_prefix"]:
            frac = random.uniform(*cfg["compress_prefix"])
            best_data, exp_attn, exp_pl = drop_nl_prefix(
                best_data, exp_attn, exp_pl, base_vocab, pad_token_id, compress_frac=frac)

        traj_mask, abs_mask = build_masks(best_data, exp_attn, exp_pl, base_vocab)

        # ---- Forward pass (with grad) ----
        outputs = model(input_ids=best_data, attention_mask=exp_attn,
                        memory_span_abs=mem_abs, memory_span_traj=cfg["mem_span_traj"])
        shift_logits = outputs.logits[..., :-1, :].contiguous()

        # ---- Compute losses ----
        traj_loss = compute_traj_loss_from_logits(shift_logits, best_data, traj_mask, base_vocab)
        abs_loss  = compute_abs_loss_from_logits(shift_logits, best_data, abs_mask, base_vocab)

        corrupted = corrupt_abstract_tokens(
            best_data, base_vocab, total_vocab,
            method=cfg["corrupt_method"], corrupt_ratio=cfg["corrupt_ratio"],
        )
        corrupt_traj = compute_corrupted_traj_loss(model, corrupted, exp_attn, traj_mask, base_vocab)
        hinge_loss = (cfg["gamma"] + traj_loss - corrupt_traj).clamp(min=0)

        loss = (
            cfg["alpha_traj"]        * traj_loss
          + cfg["alpha_contrastive"] * hinge_loss
          + cfg["alpha_abs"]         * abs_loss
        ) / cfg["grad_accum"]

        loss.backward()
        del outputs, shift_logits

        # ---- Optimizer step ----
        accum_count += 1
        if accum_count % cfg["grad_accum"] == 0:
            if cfg["max_grad_norm"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        # ---- Logging ----
        if (batch_idx + 1) % cfg["log_every"] == 0:
            total_loss = loss.item() * cfg["grad_accum"]
            elapsed = time.time() - t_start
            frac_done = max(global_step, 1) / max(total_steps, 1)
            eta = elapsed / frac_done * (1 - frac_done) if frac_done > 0 else 0
            eta_m, eta_s = divmod(int(eta), 60)
            eta_h, eta_m = divmod(eta_m, 60)
            peak = f"Mem: {torch.cuda.max_memory_allocated(device)/1024**3:.2f}GB" if torch.cuda.is_available() else ""

            print(f"ep {epoch + (batch_idx+1)/len(dl):.3f}/{cfg['num_epochs']} "
                  f"| step {global_step} | eta {eta_h}h{eta_m:02d}m "
                  f"| loss={total_loss:.4f} base={base_loss.item():.4f} "
                  f"traj={traj_loss.item():.4f} hinge={hinge_loss.item():.4f} "
                  f"abs={abs_loss.item():.4f} "
                  f"| K={K_this} mem={mem_abs} | lr={lr:.2e} {peak}")

            history["step"].append(global_step)
            history["loss"].append(total_loss)
            history["base_loss"].append(base_loss.item())
            history["traj_loss"].append(traj_loss.item())
            history["hinge_loss"].append(hinge_loss.item())
            history["abs_loss"].append(abs_loss.item())
            history["lr"].append(lr)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ---- Eval ----
        if global_step > 0 and global_step % cfg["eval_every"] == 0:
            model.eval()
            res = evaluate_accuracy(model, tokenizer, val_ds, device, cfg["eval_samples"])
            print(f"  === Eval step {global_step}: {res} ===")
            model.train()

    print(f"=== Epoch {epoch + 1} complete ===")

# Save history
with open(os.path.join(cfg["output_dir"], "history.json"), "w") as f:
    json.dump(history, f)
print(f"Training complete! History saved to {cfg['output_dir']}/history.json")

- [1/100 inner loop hinge loss with 17 mutations]:  0.44238901138305664
[Optimizer Step taken]
- [2/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
- [3/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
- [4/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
- [5/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
[Optimizer Step taken]
- [6/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
- [7/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
- [8/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
- [9/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
[Optimizer Step taken]
- [10/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
- [11/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
- [12/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
- [13/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
[Op

KeyboardInterrupt: 

In [ ]:
# Notes / Observations:
#
# Obs #1. Inner loop + dynamic corrupted + hinge only → zero hinge in 10 steps
# Obs #2. Inner loop + static corrupted + combo loss → hinge decreases then increases
# Obs #3. Anchor loss does not benefit SoRL training — removed

In [ ]:
# ============================================================
# Inner Monologue Visualization (post-training)
# ============================================================
from data.pt_dataset import _filter_traj_tokens

model.eval()
n_samples = 3
max_new_tokens = 128

for i in range(min(n_samples, len(val_ds))):
    item = val_ds[i]
    inp = item["input_ids"].unsqueeze(0).to(device)
    pl = item["prompt_len"]

    with torch.no_grad():
        generated = model.generate(
            input_ids=inp[:, :pl],
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            K=cfg["K"],
        )

    gen_tokens = generated[0, pl:]
    parts = []
    for tid in gen_tokens:
        t = tid.item()
        if t == tokenizer.eos_token_id:
            break
        if t >= base_vocab:
            parts.append(f"[A{t - base_vocab}]")
        else:
            parts.append(tokenizer.decode([t]))

    question = tokenizer.decode(inp[0, :pl], skip_special_tokens=True)
    traj = _filter_traj_tokens(generated, base_vocab)
    nl_text = tokenizer.decode(traj[0][pl:], skip_special_tokens=True)

    print(f"{'='*80}")
    print(f"Sample {i+1}: {question[:200]}...")
    print(f"\n--- Inner Monologue ---")
    print("".join(parts)[:500])
    print(f"\n--- NL-only ---")
    print(nl_text[:500])
    print()

model.train()
print("Done.")

In [ ]:
# ============================================================
# Eval Accuracy (optional)
# ============================================================
model.eval()
res = evaluate_accuracy(model, tokenizer, val_ds, device, cfg["eval_samples"])
print(res)